# 3. First XGBoost model

I want to check whether a small model can tell unpaired jets from LLP jets.
This is a first test on our files, not a finished BIB tagger.

I use the stored ATLAS jet time. I do not change the cell times.
The two earlier notebooks are unchanged.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "analysis/__init__.py").is_file())
sys.path.insert(0, str(PROJECT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT / "analysis/.cache/matplotlib"))

import pandas as pd
from IPython.display import display, Image, Markdown
from analysis.training.train_xgboost import run_experiment, score_saved_model

OUT = PROJECT / "analysis/outputs/xgboost_baseline"
RETRAIN = False  # Set True to rebuild the tables and train again.
if RETRAIN or not (OUT / "metrics.json").is_file():
    run_experiment(PROJECT, OUT)

results = json.loads((OUT / "metrics.json").read_text())
audit = json.loads((OUT / "data_audit.json").read_text())
print("XGBoost version:", results["versions"]["xgboost"])
print("Source files checked:", len(audit["files"]))
print("All source hashes checked:", all(f["hash_verified"] for f in audit["files"]))

## Which jets did I use?

- **Unpaired = 1:** the working BIB label from the meeting. Some contamination can remain.
- **LLP = 0:** only jets with `jet_is_llp_matched == 1`. Other jets in LLP events are left out.
- Both classes: $p_T \geq 30$ GeV, $|\eta| < 1.7$, and $0 < f_{\mathrm{Tile}} < 1$.

LLP matching already needs LAr and Tile cells. The last cut gives both classes mixed calorimeter support; it does not reproduce every matching rule.
These results apply only to this selected region, not all jets. The table shows how many jets remain after each step.

In [ ]:
cutflow = pd.read_csv(OUT / "cutflow.csv")
stages = ["all_jets", "truth_selection", "finite_pt_eta_time", "pt_cut", "eta_cut", "mixed_calorimeters"]
display(cutflow.pivot(index="stage", columns="sample", values="jets").reindex(stages))

## Keep the test separate

About 60% of events train the model, 20% choose when to stop and where to put the score cut, and 20% test the result.
All jets from the same event stay together.

The cut is chosen to keep at least 95% of the **validation** LLP jets. Test retention can differ.
Event IDs, truth flags, filenames and the existing BIB score never go into the model.

In [ ]:
display(pd.read_csv(OUT / "split_summary.csv"))
print("Events shared between splits:", audit["split"]["overlapping_event_groups"])

## What goes into XGBoost?

| Input | Meaning |
|---|---|
| `jet_time_ATLAS` | Standard ATLAS jet time, in ns |
| `jet_time_spread` | Spread of the stored cell times, in ns |
| `jet_time_sigma` | Estimated uncertainty on jet time, in ns |
| `jet_n_clusters` | Number of clusters in the jet |
| `jet_TileEnergyFrac` | Fraction of energy in Tile |

XGBoost adds small decision trees, one after another. Each new tree tries to improve the earlier guesses.
The output is a score: high means more BIB-like. It is not a calibrated physical probability.

I also train controls: timing alone, $p_T$ and $|\eta|$ alone, and the main inputs without time uncertainty.
The last control and the balancing study below were added after the initial comparison, so they are exploratory.

In [ ]:
settings = results["training_parameters"]
print("Maximum trees:", settings["n_estimators"])
print("Maximum tree depth:", settings["max_depth"])
print("Learning rate:", settings["learning_rate"])
print("Stop after 40 rounds without better validation loss.")
print("Saved trees in the original main model:", results["models"]["XGBoost"]["saved_trees"])

def result_table(models):
    rows = []
    for name, model in models.items():
        m = model["test"]
        rows.append({"Model": name, "Test jets": m["n_jets"], "ROC AUC": round(m["roc_auc"], 3),
                     "Unpaired tagged (%)": round(100 * m["bib_tag_fraction"], 1),
                     "LLP kept (%)": round(100 * m["llp_keep_fraction"], 1)})
    return pd.DataFrame(rows).set_index("Model")

display(result_table(results["models"]))

## Why the first high score is not enough

The kinematics-only control also works very well. The samples have very different $p_T$ distributions.
Leaving $p_T$ out of the main model does not fully fix this: timing uncertainty and cluster count can still depend on energy.

I therefore make a smaller check with equal numbers from both classes in narrow $p_T$/absolute-$\eta$ bins.
Training defines the allowed bins. Sampling stays inside each original split and uses no timing or model scores.
This reduces the difference; it does not remove every difference between MC and real data.

In [ ]:
MATCHED = OUT / "matched_kinematics"
matched = results["matched_kinematics"]
matching_audit = json.loads((MATCHED / "matching_audit.json").read_text())
display(pd.DataFrame(matching_audit["event_summary"]))
display(Image(filename=str(MATCHED / "kinematics_before_after.png"), width=900))
display(result_table(matched["models"]))

## Read the smaller check

**Unpaired tagged** means removed as BIB. **LLP kept** means not removed.
ROC AUC measures ranking: 0.5 is random and 1 is perfect. It is not the efficiency at our chosen cut.

The intervals below resample whole test events at the fixed score cut. They do not include uncertainty from training, choosing the cut, label contamination or MC/data differences.

In [ ]:
m = matched["models"]["XGBoost"]["test"]
print(f"Test sample: {m['class_counts']['BIB']} unpaired jets and {m['class_counts']['LLP']} LLP jets.")
for key, label in [("bib_tag_fraction", "Unpaired tagged"), ("llp_keep_fraction", "LLP kept")]:
    interval = m[key + "_ci95"]
    if interval is None:
        print(f"{label}: {100*m[key]:.1f}%; no informative bootstrap interval here.")
    else:
        lo, hi = interval
        print(f"{label}: {100*m[key]:.1f}% (event-bootstrap interval {100*lo:.1f}–{100*hi:.1f}%)")
display(Image(filename=str(MATCHED / "test_discrimination.png"), width=1000))

## The delayed region is still a problem

Our actual search needs to reject BIB that looks delayed, not just easy early jets.
Here I check ATLAS jet time $> 3$ ns using the **same frozen cut**.
This is a diagnostic region, not a proposed final search cut.

In [ ]:
d = matched["models"]["XGBoost"]["test_regions"]["delayed_time_gt_3ns"]
c = d["confusion"]
print(f"Delayed unpaired tagged: {c['BIB_tagged_BIB']} / {d['class_counts']['BIB']}")
print(f"Delayed LLP kept: {c['LLP_kept_as_LLP']} / {d['class_counts']['LLP']}")
print("Only a few delayed unpaired jets remain. Overall AUC does not show that this region works.")
display(Image(filename=str(MATCHED / "test_kinematic_dependence.png"), width=1000))

## Two example jets

One test jet from each label, chosen by row order. I do not pick them by their scores.
The saved balanced-check model uses the five inputs listed above.

In [ ]:
jets = pd.read_csv(MATCHED / "matched_jets.csv.gz")
examples = jets.loc[jets.split.eq("test")].groupby("label", sort=True).head(1).copy()
pred = score_saved_model(examples, MATCHED)
columns = ["sample", "jet_pt_GeV", "jet_eta", "jet_time_ATLAS", "jet_time_spread",
           "jet_time_sigma", "jet_n_clusters", "jet_TileEnergyFrac"]
display(examples[columns].join(pred).round(4))
print("BIB score cut:", m["threshold"])
print("Selected model:", MATCHED / "xgboost.ubj")

## What I would take to the meeting

I have a reproducible training run, a separate event test, and saved models.
The strong first result is partly a sample-spectrum effect. The smaller balanced check is more informative.
**It still does not demonstrate useful delayed-BIB rejection at this cut.**

Next I would ask Emil to confirm the production/matching definitions and provide more delayed BIB examples, another run and a usable collision ntuple.
Then I would add checked timing-position features from jet constituents and reserve a new test sample.

Full counts, model settings and limitations are in [run_notes.md](outputs/xgboost_baseline/run_notes.md) and `metrics.json`.
Training code: [train_xgboost.py](train_xgboost.py). Models and data-derived outputs stay outside Git.